[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%208/L15_Answering_The_Customer.ipynb)

# Answering the Customer
### ISBA 2411 · Week 8 · Lecture 15

Lecture 13 built a copilot that **read** the inbox. Lecture 14 asked what our own labelled
data would buy. Tonight it **answers the customer**, using Cobalt's own help centre, and shows
where every claim came from.

> **This notebook is not a follow-along.** The lecture is watch-only. This exists so you can read
> how the demo works and rebuild it yourself afterwards.

The whole system is four moving parts, and you have already met three of them:

| part | what it is | from |
|---|---|---|
| encoder | `all-MiniLM-L6-v2` | Lecture 13 |
| reranker | `ms-marco-MiniLM-L-6-v2` | new tonight |
| generator | `Qwen2.5-1.5B-Instruct` | new tonight |
| interface | Streamlit | new tonight |

Nothing here is trained or fine-tuned. This is **rung 3** of the ladder from Lecture 14.

---
## Setup

In [ ]:
%%capture
%pip install -q sentence-transformers transformers streamlit

#### ▶ STEP 1 &middot; Load Cobalt's help centre

In [ ]:
# -------- STEP 1 · Load Cobalt's help centre --------
import json, urllib.request, numpy as np, torch, textwrap
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
kb = json.loads(urllib.request.urlopen("https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/data/cobalt_kb.json").read())

print(f"{len(kb)} chunks from {len(set(d['doc_id'] for d in kb))} help articles")
print(f"chunk length: {min(d['words'] for d in kb)} to {max(d['words'] for d in kb)} words")
for d in kb[:6]:
    print(f"   {d['title'][:34]:36} / {d['section']}")

The help centre is small on purpose: seven articles, twenty-four chunks. **Two of the eight
ticket categories from Lecture 13 have no article at all**, so the system has genuine gaps to
run into rather than staged ones.

#### ▶ STEP 2 &middot; Look at one chunk

In [ ]:
# -------- STEP 2 · Look at one chunk --------
d = kb[1]
print(f"title   {d['title']}")
print(f"section {d['section']}")
print(f"words   {d['words']}\n")
print(textwrap.fill(d['text'], 92))

**Note what the chunk boundary is.** One chunk = one section of one article. We did not cut the
documents into fixed 500-character blocks; we split on the structure the author already wrote.
That is the recommendation from the chunking slide, and it is why every chunk here is a coherent
answer to something rather than half of two answers.

#### ▶ STEP 3 &middot; Embed the chunks once

In [ ]:
# -------- STEP 3 · Embed the chunks once --------
enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

# index the TITLE and SECTION alongside the body: the heading carries a lot of signal
X = enc.encode([f"{d['title']}. {d['section']}. {d['text']}" for d in kb],
               normalize_embeddings=True, batch_size=32)
print("index:", X.shape, "  one row per chunk, 384 numbers each")
print("this is the same encoder, and the same idea, as Lecture 13's search")

#### ▶ STEP 4 &middot; Pass 1: retrieve

In [ ]:
# -------- STEP 4 · Pass 1: retrieve --------
TICKET = "Our SSO through Okta stopped working after the weekend. Nobody on our team can sign in."

sims = X @ enc.encode([TICKET], normalize_embeddings=True)[0]
first_pass = list(np.argsort(sims)[::-1][:8])

print("PASS 1, fast and crude. Cosine similarity against all 24 chunks:\n")
for r, i in enumerate(first_pass, 1):
    print(f"  {r}. {sims[i]:.3f}  {kb[i]['title'][:30]:32} / {kb[i]['section']}")

#### ▶ STEP 5 &middot; Pass 2: rerank

In [ ]:
# -------- STEP 5 · Pass 2: rerank --------
rr = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)

# the reranker reads the question and the passage TOGETHER, which the encoder never does
scores = rr.predict([(TICKET, kb[i]["text"]) for i in first_pass])
second_pass = sorted(zip(first_pass, scores), key=lambda t: -t[1])

print("PASS 2, slow and careful. Only 8 pairs, so we can afford it:\n")
for r, (i, s) in enumerate(second_pass, 1):
    print(f"  {r}. {s:+6.2f}  {kb[i]['title'][:30]:32} / {kb[i]['section']}")

#### ▶ STEP 6 &middot; See what reranking changed

In [ ]:
# -------- STEP 6 · See what reranking changed --------
before = [kb[i]['section'] for i in first_pass][:3]
after  = [kb[i]['section'] for i, _ in second_pass][:3]
print(f"{'top 3 after retrieval':38}   top 3 after reranking")
print("-" * 84)
for b, a in zip(before, after):
    flag = "" if b == a else "   <- changed"
    print(f"{b[:36]:38}   {a[:32]}{flag}")

✅ **What just happened.** The encoder compares the question to each chunk *separately*: it turned
the question into 384 numbers, turned each chunk into 384 numbers, and measured distance. It never
saw the two together.

The reranker does see them together, so it can judge things the encoder cannot, like whether the
passage actually answers the question rather than merely being about the same topic.

💼 **At work this means:** cheap and rough to narrow the field, expensive and careful to pick the
winner. You could not afford to rerank all 24 chunks in a large system, but reranking 8 is nothing.

#### ▶ STEP 7 &middot; Build the grounded prompt

In [ ]:
# -------- STEP 7 · Build the grounded prompt --------
PICKED = second_pass[:3]

SYSTEM = ("You are a Cobalt support agent. You may ONLY use facts from the numbered passages.\n"
          "RULES:\n"
          "1. After EVERY sentence that states a fact, put the passage number in brackets, e.g. [2].\n"
          "2. If the passages do not answer the ticket, reply with exactly this and nothing else:\n"
          "   NO_ANSWER\n"
          "3. Never name a menu, setting or feature that does not appear in the passages.\n"
          "Example of a good reply:\n"
          "Go to Admin, then Identity Providers, and choose Reconnect [1]. Your signing certificate "
          "must be current or the reconnect will fail [1].")

ctx = "\n".join(f"[{n+1}] ({kb[i]['title']} / {kb[i]['section']}) {kb[i]['text']}"
                for n, (i, _) in enumerate(PICKED))
print(SYSTEM)
print("\n" + "=" * 90 + "\n")
print(f"Passages:\n{ctx}\n\nTicket: {TICKET}\n\nReply:")

**That prompt is the entire trick.** There is no training, no fine-tuning, no vector database
product. The model is exactly as downloaded. The only thing that changed is that we put three
relevant passages in front of it before asking.

Two details in the rules are there because they were **tested and needed**:

- The one-shot example. Without it, a 1.5B model ignores the citation instruction completely.
- The literal token `NO_ANSWER`. Without it, asked about a feature Cobalt does not have, the model
  cheerfully recommended enabling it.

#### ▶ STEP 8 &middot; Generate the reply

In [ ]:
# -------- STEP 8 · Generate the reply --------
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32   # fp32 on a GPU wastes half the memory
gen = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct",
                                           dtype=DTYPE).to(DEVICE).eval()

def answer(ticket, picked):
    ctx = "\n".join(f"[{n+1}] ({kb[i]['title']} / {kb[i]['section']}) {kb[i]['text']}"
                    for n, (i, _) in enumerate(picked))
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Passages:\n{ctx}\n\nTicket: {ticket}\n\nReply:"}]
    p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(p, return_tensors="pt").to(DEVICE)
    with torch.no_grad():                      # greedy: same answer every time, auditable
        out = gen.generate(**ids, max_new_tokens=160, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

reply = answer(TICKET, PICKED)
print(textwrap.fill(reply, 92))
print("\nsources:")
for n, (i, s) in enumerate(PICKED, 1):
    print(f"  [{n}] {kb[i]['title']} / {kb[i]['section']}   (rerank {s:+.2f})")

✅ **What just happened.** A grounded answer with a citation, from a model that has never seen
Cobalt. Check the bracketed number against the sources printed underneath: that is the check a
support rep does in four seconds before sending the reply.

💼 **At work this means:** the citation is not decoration. It is what makes the answer *checkable*,
and it is what lets you find the bad document afterwards when the answer turns out to be wrong.

#### ▶ STEP 9 &middot; The pipeline, and a question it cannot answer

In [ ]:
# -------- STEP 9 · The pipeline, and a question it cannot answer --------
def pipeline(ticket, k1=8, k2=3, verbose=True):
    sims = X @ enc.encode([ticket], normalize_embeddings=True)[0]
    cand = list(np.argsort(sims)[::-1][:k1])
    sc   = rr.predict([(ticket, kb[i]["text"]) for i in cand])
    picked = sorted(zip(cand, sc), key=lambda t: -t[1])[:k2]
    reply  = answer(ticket, picked)
    if verbose:
        print(f"TICKET: {ticket}\n")
        print("retrieved:")
        for n, (i, s) in enumerate(picked, 1):
            print(f"  [{n}] {s:+6.2f}  {kb[i]['section']}")
        print(f"\nREPLY: {textwrap.fill(reply, 88)}\n")
    return reply, picked

_ = pipeline("Please add dark mode. Our team works late and the white background is rough.")

✅ **The refusal, and it is the feature.** Cobalt has no dark mode, and there is no article about
it. The model was handed three passages about unrelated things and said `NO_ANSWER` rather than
inventing a setting.

Compare that with the answer on the failure slide at the start of the lecture, where a general
model confidently invented a menu path. **Same model family, same question shape. The only
difference is that this one was given documents and told to stay inside them.**

#### ▶ STEP 10 &middot; Watch a retrieval failure

In [ ]:
# -------- STEP 10 · Watch a retrieval failure --------
_ = pipeline("Sarah left last week but her account still has access. Please revoke it.")

print("=" * 90)
print("But Cobalt's help centre DOES cover this. Where did the right passage rank?\n")
sims = X @ enc.encode(["Sarah left last week but her account still has access. Please revoke it."],
                      normalize_embeddings=True)[0]
cand = list(np.argsort(sims)[::-1][:8])
sc   = rr.predict([("Sarah left last week but her account still has access. Please revoke it.",
                    kb[i]["text"]) for i in cand])
for r, (i, s) in enumerate(sorted(zip(cand, sc), key=lambda t: -t[1]), 1):
    mark = "   <-- the passage that actually answers it" if "leaves" in kb[i]["section"] else ""
    print(f"  {r}. {s:+6.2f}  {kb[i]['section']}{mark}")

✅ **This is the most useful failure in the notebook.** The help centre has a section called
*Removing access when someone leaves* which answers the ticket exactly. The reranker put it
**sixth of eight**, below three passages that are about nothing relevant, so it never reached the
model. The model then correctly refused, because what it was given genuinely did not answer.

**The generator did nothing wrong. Retrieval did.**

💼 **At work this means:** when a RAG system gives a bad answer, or refuses one it should have
answered, look at what was retrieved before you touch the prompt. Most RAG failures are retrieval
failures, and they are invisible if you only ever read the final answer. That is why the
evaluation slide insists on measuring the two halves separately.

---
## The product

Everything above is about 40 lines. The interface is another 100.

The cell below **downloads the app and starts it**. On your own machine it prints a command to
run; in Colab it starts a tunnel and prints a link you can open. The link takes about ninety
seconds to appear because the generator has to download first.

The sidebar exposes tonight's decisions as dials: whether to rerank, how many passages to send,
when to refuse, and greedy versus sampling. **Turn reranking off and re-run the SSO ticket** to
watch the second pass earn its place.

#### ▶ STEP 11 &middot; Run the Streamlit app

In [ ]:
# -------- STEP 11 · Run the Streamlit app --------
# The app runs as a SEPARATE process and loads its own copy of the models. This kernel
# is still holding the ones from the steps above, which on a Colab GPU is enough to make
# the app crash with an out-of-memory error. Give the memory back first.
import gc, os, re, subprocess, sys, time, urllib.request
for _name in ("gen", "enc", "rr", "X"):
    if _name in dir():
        exec(f"del {_name}")
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info()
        print(f"GPU freed: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")
except Exception:
    pass

RAW = "https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/"
urllib.request.urlretrieve(RAW + "Week%208/demo/app.py", "app.py")
urllib.request.urlretrieve(RAW + "data/cobalt_kb.json", "cobalt_kb.json")

# app.py expects the knowledge base one directory above it, so give it one
os.makedirs("demo", exist_ok=True)
os.replace("app.py", "demo/app.py")
print("downloaded demo/app.py and cobalt_kb.json")

if "google.colab" not in sys.modules:
    print("\nRunning locally. Start the app with:\n")
    print("    streamlit run demo/app.py\n")
    print("It opens at http://localhost:8501")
else:
    print("\nColab cannot serve a port directly, so we tunnel it. Give this ~90 seconds:")
    subprocess.run("pip -q install streamlit", shell=True)
    subprocess.run("wget -q -O cloudflared https://github.com/cloudflare/cloudflared/"
                   "releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared",
                   shell=True)
    subprocess.Popen("streamlit run demo/app.py --server.port 8501 --server.headless true "
                     "> streamlit.log 2>&1", shell=True)
    subprocess.Popen("./cloudflared tunnel --url http://localhost:8501 --no-autoupdate "
                     "> tunnel.log 2>&1", shell=True)
    url = None
    for _ in range(60):
        time.sleep(3)
        try:
            m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
            if m:
                url = m.group(0)
                break
        except FileNotFoundError:
            pass
    if url:
        print(f"\n    OPEN THIS:  {url}\n")
        print("That is the running app, not a code listing.")
    else:
        print("\nTunnel did not start. Check tunnel.log and streamlit.log.")

---
## The one idea to take with you

Lecture 13: *someone else already paid to teach a model to read.*
Lecture 14: *your own labels are a purchasing decision, and which ones you buy matters more than
how many.*

Tonight:

> **The model does not need to know your company. It needs to be handed the right paragraph at the
> moment you ask.** Retrieval is what does the handing, citations are what make the answer
> checkable, and a refusal is a feature rather than a failure.

### What to look at on Monday
1. When a RAG answer is wrong, read what was retrieved before you touch the prompt.
2. Ask what happens when the answer is not in the documents. If nobody has tested that, it invents.
3. Use greedy decoding for anything a customer will read, so the same question gives the same answer.
4. Insist on citations. An answer nobody can check is an answer nobody can defend.